In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns

# Visual formatting defaults
plt.style.use(
    'seaborn-v0_8-whitegrid'
    if 'seaborn-v0_8-whitegrid' in plt.style.available
    else 'default'
)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cbd5e1'
plt.rcParams['axes.linewidth'] = 0.8

In [ ]:
df = pd.read_csv('online_retail.csv', encoding='ISO-8859-1')

print(f'Raw Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
print('Missing entries:\n', df.isnull().sum())
df.head(3)

In [ ]:
# Strip cancellations and non-positive transactional entries
clean_df = df[
    (~df['InvoiceNo'].astype(str).str.startswith('C'))
    & (df['Quantity'] > 0)
    & (df['UnitPrice'] > 0)
].copy()

# Feature expansion
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
clean_df['TotalPrice'] = clean_df['Quantity'] * clean_df['UnitPrice']
clean_df['YearMonth'] = clean_df['InvoiceDate'].dt.to_period('M')
clean_df['Hour'] = clean_df['InvoiceDate'].dt.hour
clean_df['DayOfWeek'] = clean_df['InvoiceDate'].dt.day_name()

print(f'Clean records: {len(clean_df):,}')

In [ ]:
total_revenue = clean_df['TotalPrice'].sum()
total_orders = clean_df['InvoiceNo'].nunique()
total_customers = clean_df['CustomerID'].dropna().nunique()
avg_basket = total_revenue / total_orders

print(f'Total Revenue:     ${total_revenue:,.2f}')
print(f'Total Orders:      {total_orders:,}')
print(f'Unique Accounts:   {total_customers:,}')
print(f'Average Basket:    ${avg_basket:,.2f}')

In [ ]:
# Monthly Trajectory
monthly_rev = clean_df.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly_rev['YearMonthStr'] = monthly_rev['YearMonth'].astype(str)

# Regional Distribution
country_rev = (
    clean_df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False)
)
top_export = country_rev[country_rev.index != 'United Kingdom'].head(5)

# Temporal Rhythms
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday']
dow_rev = (
    clean_df.groupby('DayOfWeek')['TotalPrice']
    .sum()
    .reindex(dow_order)
    .dropna()
)
hourly_rev = clean_df.groupby('Hour')['TotalPrice'].sum()

# Top SKUs
top_products = (
    clean_df.groupby('Description')['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
fig1 = plt.figure(figsize=(16, 10), facecolor='#f8fafc')
fig1.text(
    0.06,
    0.95,
    'ONLINE RETAIL EXECUTIVE DASHBOARD',
    fontsize=20,
    fontweight='bold',
    color='#0f172a',
)
fig1.text(
    0.06,
    0.925,
    'Performance Diagnostics, Revenue Velocity & Operational Health (Dec 2010'
    ' - Dec 2011)',
    fontsize=11,
    color='#64748b',
)

# Metric summary cards
kpis = [
    ('TOTAL REVENUE', f'${total_revenue/1e6:.2f}M', 'Total gross billed'),
    ('TOTAL TRANSACTIONS', f'{total_orders:,}', 'Distinct invoices'),
    ('ACTIVE CUSTOMERS', f'{total_customers:,}', 'Registered accounts'),
    ('AVG BASKET VALUE', f'${avg_basket:.2f}', 'Per completed order'),
]
card_w, card_h, card_y = 0.205, 0.085, 0.82
for i, (lbl, val, sub) in enumerate(kpis):
  cx = 0.06 + i * (card_w + 0.023)
  ax = fig1.add_axes([cx, card_y, card_w, card_h], facecolor='white')
  for s in ['top', 'right', 'left', 'bottom']:
    ax.spines[s].set_color('#cbd5e1')
  ax.set_xticks([])
  ax.set_yticks([])
  ax.text(
      0.08,
      0.72,
      lbl,
      fontsize=8.5,
      fontweight='bold',
      color='#64748b',
      transform=ax.transAxes,
  )
  ax.text(
      0.08,
      0.35,
      val,
      fontsize=16,
      fontweight='bold',
      color='#0f172a',
      transform=ax.transAxes,
  )
  ax.text(
      0.08,
      0.12,
      sub,
      fontsize=7.5,
      fontweight='medium',
      color='#2563eb',
      transform=ax.transAxes,
  )

# Monthly curve
ax_m = fig1.add_axes([0.06, 0.44, 0.55, 0.33], facecolor='white')
ax_m.plot(
    monthly_rev['YearMonthStr'],
    monthly_rev['TotalPrice'] / 1e3,
    marker='o',
    color='#2563eb',
    lw=2.5,
)
ax_m.fill_between(
    monthly_rev['YearMonthStr'],
    monthly_rev['TotalPrice'] / 1e3,
    color='#93c5fd',
    alpha=0.22,
)
ax_m.set_title(
    'Monthly Revenue Trajectory (USD Thousands)',
    fontsize=11,
    fontweight='bold',
    color='#1e293b',
    loc='left',
)
ax_m.tick_params(axis='x', rotation=30, labelsize=8.5)
ax_m.grid(True, linestyle='--', alpha=0.5, color='#cbd5e1')

# International breakdown
ax_e = fig1.add_axes([0.67, 0.44, 0.28, 0.33], facecolor='white')
ax_e.barh(
    top_export.index[::-1],
    top_export.values[::-1] / 1e3,
    color=['#bfdbfe', '#93c5fd', '#60a5fa', '#3b82f6', '#1d4ed8'],
    height=0.55,
)
ax_e.set_title(
    'Top 5 Export Markets ($k)',
    fontsize=11,
    fontweight='bold',
    color='#1e293b',
    loc='left',
)
ax_e.grid(True, linestyle='--', alpha=0.5, axis='x', color='#cbd5e1')

# Day of week
ax_d = fig1.add_axes([0.06, 0.08, 0.27, 0.28], facecolor='white')
ax_d.bar(dow_rev.index, dow_rev.values / 1e3, color='#0284c7', width=0.55)
ax_d.set_title(
    'Sales by Day of Week ($k)',
    fontsize=10.5,
    fontweight='bold',
    color='#1e293b',
    loc='left',
)
ax_d.tick_params(axis='x', rotation=35, labelsize=8)
ax_d.grid(True, linestyle='--', alpha=0.5, axis='y', color='#cbd5e1')

# Hourly cadence
ax_h = fig1.add_axes([0.375, 0.08, 0.27, 0.28], facecolor='white')
ax_h.plot(
    hourly_rev.index,
    hourly_rev.values / 1e3,
    marker='s',
    color='#d97706',
    lw=2,
    markersize=4,
)
ax_h.fill_between(
    hourly_rev.index, hourly_rev.values / 1e3, color='#fef3c7', alpha=0.5
)
ax_h.set_title(
    'Intraday Demand Velocity ($k)',
    fontsize=10.5,
    fontweight='bold',
    color='#1e293b',
    loc='left',
)
ax_h.grid(True, linestyle='--', alpha=0.5, color='#cbd5e1')

# Domestic vs Export split
ax_p = fig1.add_axes([0.69, 0.08, 0.26, 0.28], facecolor='white')
uk_rev = country_rev.loc['United Kingdom']
ax_p.pie(
    [uk_rev, country_rev.sum() - uk_rev],
    labels=['Domestic (UK)', 'International'],
    autopct='%1.1f%%',
    colors=['#1e40af', '#38bdf8'],
    startangle=140,
    explode=(0.04, 0),
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2),
)
ax_p.set_title(
    'Market Contribution Split',
    fontsize=10.5,
    fontweight='bold',
    color='#1e293b',
    loc='left',
)
plt.show()

In [ ]:
fig2, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#f8fafc')
fig2.suptitle(
    'PRODUCT DYNAMICS & CUSTOMER COHORT DEEP DIVE',
    fontsize=18,
    fontweight='bold',
    color='#0f172a',
    y=0.96,
)

# Top SKUs
sns.barplot(
    x=top_products.values / 1e3,
    y=[
        d[:25] + '...' if len(str(d)) > 25 else str(d)
        for d in top_products.index
    ],
    ax=axes[0, 0],
    palette='Blues_r',
)
axes[0, 0].set_title(
    'Top 10 Products by Revenue ($k)',
    fontweight='bold',
    fontsize=11,
    color='#1e293b',
)

# Price elasticity (Log-Log)
sample_scatter = clean_df.sample(n=min(6000, len(clean_df)), random_state=42)
axes[0, 1].scatter(
    sample_scatter['UnitPrice'],
    sample_scatter['Quantity'],
    alpha=0.35,
    color='#0284c7',
    s=18,
)
axes[0, 1].set_xscale('log')
axes[0, 1].set_yscale('log')
axes[0, 1].set_title(
    'Price Elasticity: Unit Price vs Quantity (Log-Log)',
    fontweight='bold',
    fontsize=11,
    color='#1e293b',
)

# Customer Lifetime Spend Distribution
cust_spend = clean_df.groupby('CustomerID')['TotalPrice'].sum()
axes[1, 0].hist(
    cust_spend[cust_spend < 10000], bins=40, color='#6366f1', edgecolor='white'
)
axes[1, 0].set_title(
    'Customer Spend Distribution (< $10k)',
    fontweight='bold',
    fontsize=11,
    color='#1e293b',
)

# Top VIP Accounts
top_vip = cust_spend.sort_values(ascending=False).head(10)
sns.barplot(
    x=top_vip.values / 1e3,
    y=[f'ID: {int(c)}' for c in top_vip.index],
    ax=axes[1, 1],
    palette='crest_r',
)
axes[1, 1].set_title(
    'Top 10 VIP Accounts by Spend ($k)',
    fontweight='bold',
    fontsize=11,
    color='#1e293b',
)

for ax in axes.flat:
  ax.set_facecolor('white')
  ax.grid(True, linestyle='--', alpha=0.5, color='#cbd5e1')
plt.subplots_adjust(top=0.90, hspace=0.3, wspace=0.25)
plt.show()